In [11]:
import pandas as pd
import os
# os.chdir("../")
from src.customer_churn.logging.logger import logging
from src.customer_churn.exception.exception import CustomerChurnException
from src.customer_churn.entity.artifact_entity import DataCleaningArtifacts, FeatureEngineeringArtifacts
from src.customer_churn.config.configuration import FeatureEngineeringConfig
import time

## 1. Import the libraries:

In [26]:
test_clipped = pd.read_parquet("Artifacts/data_cleaning/data_unclipped/test.parquet")
test_clipped.head(1)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,order_issue,quantity_abs
0,508972,22605,WOODEN CROQUET GARDEN SET,2,2010-05-19 11:39:00,12.75,15727.0,United Kingdom,0,2


In [1]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model\\research'

# 2. Import the training data:

In [4]:
train = pd.read_parquet("Artifacts/data_cleaning/data_clipped/train.parquet")

In [5]:
train.head(2)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,order_issue,quantity_abs,quantity_abs_clipped,price_clipped
0,532371,21212,PACK OF 72 RETROSPOT CAKE CASES,48,2010-11-11 17:55:00,0.55,15358.0,United Kingdom,0,48,48,0.55
1,532810,22560,TRADITIONAL MODELLING CLAY,1,2010-11-14 15:23:00,1.25,17545.0,United Kingdom,0,1,1,1.25


<!-- ## 3.2 Transform into function: -->

In [14]:
# def define_churn_label(df:pd.DataFrame, churn_cutoff_date:str, data_name:str) -> pd.DataFrame:
#     """
#     Adds a new feature churn label: 1(Churn) & 0(Retain)
#     Parameters:
#     -----------
#     df: DataFrame at customer level aggregation where churn feature needs to be added
#     churn_cutoff_date: The churn window end date
#     """
#     # Set the observation and churn date:
#     chun_cutoff_date = pd.to_datetime(churn_cutoff_date)
#     observation_date = chun_cutoff_date - pd.DateOffset(months=3)
#     logging.info(f"Churn window frame from {observation_date + pd.DateOffset(days=1)} : {churn_cutoff_date}")
#     logging.info(f"Observation window is set till: {observation_date}")    

#     # Split the data into observation and churn window dataframe:
#     observation_df = df[df['invoicedate'] <= observation_date].copy()
#     churn_df = train[(df['invoicedate'] > observation_date) & (df['invoicedate'] <= chun_cutoff_date)].copy()
#     customer_ids_from_observation_df = set(observation_df['customerid'].unique())

#     # Logs:
#     logging.info(f"Observation df of shape: ({observation_df.shape}) and churn df of shape: ({churn_df.shape}) created.")
#     logging.info(f"Observation df has invoice date of min : {observation_df['invoicedate'].min()} and max:       {observation_df['invoicedate'].max()} | Churn df has invoice date of min : {churn_df['invoicedate'].min()} and max:{churn_df['invoicedate'].max()}")

#     # Labelling churn:
#     observation_df['churn'] = (~observation_df['customerid'].isin(customer_ids_from_churn_df)).astype(int)
    
#     # Logs:
#     logging.info(f"churn feature created | The count of churn(1) and retain(0) is:\n{observation_df['churn'].value_counts()}")
    
#     return df
    
    

# 3. Start Creating New Features:

Define the `churn_cutoff_date`, later it will be used from `params.yaml` file:

In [7]:
churn_cutoff_date = '2011-03-31'
churn_cutoff_date = pd.to_datetime(churn_cutoff_date)
observation_date = churn_cutoff_date - pd.DateOffset(months=3)

In [14]:
print(f"The observation window will be until: {observation_date}\n")
print(f"The churn window will be from: {observation_date + pd.DateOffset(days=1)} until: {churn_cutoff_date}")

The observation window will be until: 2010-12-31 00:00:00

The churn window will be from: 2011-01-01 00:00:00 until: 2011-03-31 00:00:00


Pull the transaction from the given dataset that `falls under observation window:`

In [15]:
observation_df = train[train['invoicedate'] <= observation_date].copy()

In [17]:
observation_df.shape

(341359, 12)

In [19]:
min_date, max_date = observation_df['invoicedate'].min(), observation_df['invoicedate'].max()
print(f"The transactiosn in the observation window ranges between: {min_date} : {max_date}")

The transactiosn in the observation window ranges between: 2009-12-01 07:45:00 : 2010-12-23 16:06:00


## 1. Product and customer level metrics:

### Defining the product/stocklevel metrics:

### 1.1 Add `Total number of orders, total number of order issues, and total order issue rate` at `stockcode` level:

Here, stock level metrics will be calculated for each unique stockcode, those are:
1. **total_n_orders:** shows the popularity of the product in general
2. **total_n_order_issues:** the total issues the products faced
3. **total_order_issue_rate:** the rate at which the consumer faces the issues when buying certain products

In [20]:
stock_code_df = observation_df[['invoice', 'stockcode', 'quantity', 'order_issue']].copy()

In [21]:
print(f"There are a total of {stock_code_df['stockcode'].nunique():,} unique products.")

There are a total of 4,009 unique products.


Adding `total_n_orders` and `total_n_order_issues per stockcode`:

In [22]:
stock_code_df.head(1)

,invoice,stockcode,quantity,order_issue
0,532371,21212,48,0


In [35]:
stock_code_df_aggregated = stock_code_df.groupby('stockcode').agg(
    total_n_orders_of_prod = ('invoice', 'nunique'),
    total_n_order_issues_with_prod = ('order_issue', 'sum')).reset_index()

In [36]:
stock_code_df_aggregated.head(2)

,stockcode,total_n_orders_of_prod,total_n_order_issues_with_prod
0,10002,221,6
1,10080,5,0


Adding `total_order_issue_rate`:

In [37]:
stock_code_df_aggregated['total_order_issue_rate_w_prod'] = stock_code_df_aggregated['total_n_order_issues_with_prod'] / stock_code_df_aggregated['total_n_orders_of_prod']

In [38]:
stock_code_df_aggregated.head(2)

,stockcode,total_n_orders_of_prod,total_n_order_issues_with_prod,total_order_issue_rate_w_prod
0,10002,221,6,0.027149
1,10080,5,0,0.000000


### 1.2 Product level metrics for each product bought by customer:

Here, first a new df will be created which will hold each customerid and the stockcode of the product they purchased, then it will be joined with 
`stock_code_df` to pull the stockcode/product level metrics that was calculated earlier:

`product_per_customer` table holds the information about the `customerid` and all the `stockcode` of the products they have purchased:

In [39]:
product_per_customer = observation_df[['customerid', 'stockcode']].drop_duplicates()

In [40]:
product_per_customer.shape

(238842, 2)

In [41]:
product_per_customer.head(2)

,customerid,stockcode
0,15358.0,21212
1,17545.0,22560


The `customer_and_prod_lvl_metrics` and `stock_code_df_aggregated` dataframe will be joined on `stockcode` with 'right' join
to pull the product level metrics for all the products bought by customers which are identified by their `stockcode` 

In [42]:
customer_and_prod_lvl_metrics = stock_code_df_aggregated.merge(right=product_per_customer,
                                                               how='right',
                                                              on='stockcode')

In [43]:
customer_and_prod_lvl_metrics.head(2)

,stockcode,total_n_orders_of_prod,total_n_order_issues_with_prod,total_order_issue_rate_w_prod,customerid
0,21212,1280,13,0.010156,15358.0
1,22560,276,2,0.007246,17545.0


In [44]:
customer_risk_at_product_lvl = customer_and_prod_lvl_metrics.groupby(['customerid']).agg(
    avg_stockcode_issue_rate=('total_order_issue_rate_w_prod', 'mean'),
    max_stockcode_issue_rate=('total_order_issue_rate_w_prod', 'max'),
    total_orders_made_for_stock = ('total_n_orders_of_prod', 'sum')
).reset_index()

In [45]:
customer_risk_at_product_lvl.head(2)

,customerid,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock
0,12346.0,0.098834,1.053333,6700
1,12347.0,0.024199,0.102564,11738


## 2. Creating customer level aggregated metrics:

In [51]:
customer_data = observation_df[['invoice', 'quantity', 'invoicedate', 'customerid', 'country', 'order_issue',
                               'quantity_abs_clipped', 'price_clipped']].copy()

In [52]:
customer_data['total_amt_spent'] = customer_data['quantity'] * customer_data['price_clipped']

In [53]:
customer_data.head()

,invoice,quantity,invoicedate,customerid,country,order_issue,quantity_abs_clipped,price_clipped,total_amt_spent
0,532371,48,2010-11-11 17:55:00,15358.0,United Kingdom,0,48,0.55,26.40
1,532810,1,2010-11-14 15:23:00,17545.0,United Kingdom,0,1,1.25,1.25
2,535267,1,2010-11-25 13:36:00,14487.0,United Kingdom,0,1,3.75,3.75
3,516785,24,2010-07-23 09:30:00,12755.0,Japan,0,24,2.95,70.80
4,528089,1,2010-10-20 13:12:00,14449.0,United Kingdom,0,1,0.65,0.65


### 2.1 Adding the `customer level metrics` based on their purchasing data:

In [54]:
customer_level_metrics = customer_data.groupby(by='customerid').agg(
    # 1. Frequency: Total num of unique orders made per customer:
    order_frequency = ('invoice', 'nunique'),
    # 2. Monetary: Total amount spent by each customer:
    total_monetary_value = ('total_amt_spent', 'sum'),
    # 3. Total quantity(abs value): involves purchased and had intentions to acquire:
    total_quantity_abs = ('quantity_abs_clipped', 'sum'),
    # 4. Total issues in the orders made (order cancellation or products returned):
    total_order_issues = ('order_issue', 'sum'),
    # 5. First and last purchase made:
    first_purchase = ('invoicedate', 'min'),
    last_purchase = ('invoicedate', 'max'),
    # 6. Country: most probably has one country per customer but still a condition is defined:
    country = ('country', lambda c: c.mode()[0] if not c.mode().empty else 'Unspecified')
).reset_index()


customer_level_metrics['avg_order_value'] = customer_level_metrics['total_monetary_value'] / customer_level_metrics['order_frequency']
customer_level_metrics['avg_quantity_per_order'] = customer_level_metrics['total_quantity_abs'] / customer_level_metrics['order_frequency']
customer_level_metrics['customer_order_issue_rate'] = customer_level_metrics['total_order_issues'] / customer_level_metrics['order_frequency']
customer_level_metrics['recency'] = (churn_cutoff_date - customer_level_metrics['last_purchase']).dt.days
customer_level_metrics['tenure'] = (churn_cutoff_date - customer_level_metrics['first_purchase']).dt.days

In [55]:
customer_risk_at_product_lvl.head(1)

,customerid,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock
0,12346.0,0.098834,1.053333,6700


In [56]:
customer_level_metrics.drop(columns=['first_purchase', 'last_purchase'], inplace=True)

# 4. Merge two dataframes:

In [60]:
final_customer_lvl_features = customer_level_metrics.merge(customer_risk_at_product_lvl,
                                                 on='customerid',
                                                 how='left')

In [61]:
final_customer_lvl_features.head(2)

,customerid,order_frequency,total_monetary_value,total_quantity_abs,total_order_issues,country,avg_order_value,avg_quantity_per_order,customer_order_issue_rate,recency,tenure,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock
0,12346.0,12,92.76,67,12,United Kingdom,7.73,5.583333,1.0,177,471,0.098834,1.053333,6700
1,12347.0,2,1189.12,693,0,Iceland,594.56,346.500000,0.0,113,150,0.024199,0.102564,11738


In [62]:
final_customer_lvl_features.head(2)

,customerid,order_frequency,total_monetary_value,total_quantity_abs,total_order_issues,country,avg_order_value,avg_quantity_per_order,customer_order_issue_rate,recency,tenure,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock
0,12346.0,12,92.76,67,12,United Kingdom,7.73,5.583333,1.0,177,471,0.098834,1.053333,6700
1,12347.0,2,1189.12,693,0,Iceland,594.56,346.500000,0.0,113,150,0.024199,0.102564,11738


Now, this final dataframe will be used for further processes.

# 5. Define the `churn`:

- The transactions after the date `2011-03-31` are already dropped during data ingestion.
- The **observation window** would be `2009-12-01`(this is the first purchase invoice date recorded in the dataset) till `2010-12-31`
- The **churn window** would be (**latest date or the max date from the observation window + 3 Months**) i.e. `2011-01-01` to `2011-03-31`

#### Logic implemented for defining the churn label:
- The churn window will be used as a filter to define the churn(1) or retained(0)
- If customers from `the observation window` didn't make a single purchase in **the churn window** then the customers will be labelled as  **1(churn) else 0(retained)**

<hr>

#### Steps involved:
1. A new `dataframe` will be created i.e. `df with churn window(churn_df)`
2. Get the unique `customerid` from `churn_df`
3. Create a new feature `churn` on the customer level and product level aggregated and merged data i.e. `final_customer_lvl_features`, if `customerid` is in `customerid` of `churn_df` then **0 (retained) else 1(churn).**

## 5.1 Add the churn feature:

In [67]:
print(f"The observation window will be until: {observation_date}\n")
print(f"The churn window will be from: {observation_date + pd.DateOffset(days=1)} until: {churn_cutoff_date}")

The observation window will be until: 2010-12-31 00:00:00

The churn window will be from: 2011-01-01 00:00:00 until: 2011-03-31 00:00:00


In [69]:
# observation_df = train[train['invoicedate'] <= observation_date].copy()
churn_df = train[(train['invoicedate'] > observation_date) & (train['invoicedate'] <= churn_cutoff_date)].copy()

In [70]:
print(f"The min invoice date in churn df: {churn_df['invoicedate'].min()} and the max is: {churn_df['invoicedate'].max()}")

The min invoice date in churn df: 2011-01-04 10:00:00 and the max is: 2011-03-30 17:33:00


In [71]:
retained_customer_ids = set(churn_df['customerid'].unique())
print(f"There are a total of: {len(retained_customer_ids)} customer IDs who made the transaction in churn window.")

There are a total of: 1781 customer IDs who made the transaction in churn window.


In [72]:
final_customer_lvl_features['churn'] = (~final_customer_lvl_features['customerid'].isin(retained_customer_ids)).astype(int)

In [73]:
final_customer_lvl_features.head(2)

,customerid,order_frequency,total_monetary_value,total_quantity_abs,total_order_issues,country,avg_order_value,avg_quantity_per_order,customer_order_issue_rate,recency,tenure,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock,churn
0,12346.0,12,92.76,67,12,United Kingdom,7.73,5.583333,1.0,177,471,0.098834,1.053333,6700,0
1,12347.0,2,1189.12,693,0,Iceland,594.56,346.500000,0.0,113,150,0.024199,0.102564,11738,0


In [74]:
final_customer_lvl_features['churn'].value_counts()

churn
1    2966
0    1416
Name: count, dtype: int64

In [75]:
final_customer_lvl_features.shape

(4382, 15)

# 6. Combine everything and create a `FeatureEngineering` class:

In [28]:
class FeatureEngineering:
    def __init__(self, feature_engineering_config: FeatureEngineeringConfig, 
                 data_cleaning_artifacts:DataCleaningArtifacts,
                churn_cutoff_date: str):
        try:
            self.churn_cutoff_date = pd.to_datetime(churn_cutoff_date)
            self.observation_date = self.churn_cutoff_date - pd.DateOffset(months=3)
            self.feature_engineering_config = feature_engineering_config
            self.clipped_train_file_path = self.feature_engineering_config.clipped_train_file_path
            self.clipped_test_file_path = self.feature_engineering_config.clipped_test_file_path
            self.unclipped_train_file_path = self.feature_engineering_config.unclipped_train_file_path
            self.unclipped_test_file_path = self.feature_engineering_config.unclipped_test_file_path
            self.data_cleaning_artifacts = data_cleaning_artifacts
            logging.info(f"FeatureEngineering Class initialization success!")
        except Exception as e:
            raise CustomerChurnException(e, sys)
            

        def create_observation_df(self, df:pd.DataFrame) -> pd.DataFrame:
            """
            Creates a new df that holds the transactions for the observation window:
            """
            try:
                logging.info("Creating observation dataframe:")
                observation_df = df[df['invoicedate'] <= self.observation_date].copy()
                min_date, max_date = observation_df['invoicedate'].min(), observation_df['invoicedate'].max()
                logging.info(f"The shape of the observation dataframe is: {observation_df.shape}.")
                logging.info(f"The transactiosn in the observation window ranges between: {min_date} : {max_date}")
                logging.info("Creation of observation dataframe success!")
                return observation_df
            except Exception as e:
                raise CustomerChurnException(e, sys)
                

        def create_stockcode_lvl_metrics(self, observation_df:pd.DataFrame) -> pd.DataFrame:
            """
            Takes in the observation dataframe and creates the product/stockcode level metrics:
            """
            try:
                logging.info("Creating the product/stockcode level metrics:")
                stock_code_df= observation_df[['invoice', 'stockcode', 'quantity', 'order_issue']].copy()
                logging.info(f"There are a total of {stock_code_df['stockcode'].nunique():,} unique products.")
                
                # Adding total_n_orders and total_n_order_issues per stockcode:
                stock_code_df_aggregated = stock_code_df.groupby('stockcode').agg(
                    total_n_orders_of_prod = ('invoice', 'nunique'),
                    total_n_order_issues_with_prod = ('order_issue', 'sum')).reset_index()

                # Adding total_order_issue_rate:
                stock_code_df_aggregated['total_order_issue_rate_w_prod'] = (stock_code_df_aggregated['total_n_order_issues_with_prod'] / 
                stock_code_df_aggregated['total_n_orders_of_prod'])
                logging.info("Success for creating the product/stockcode level metrics:")
                return stock_code_df_aggregated                
            except Exception as e:
                raise CustomerChurnException(e, sys)


        def create_customer_risk_at_product_lvl(self, observation_df:pd.DataFrame, 
                                                stock_code_df_aggregated:pd.DataFrame) -> pd.DataFrame:
            """
            Takes in the observation dataframe and creates the customer level metrics:
            """
            try:
                logging.info("Creating the customer level risk for the products they buy:")
                product_per_customer = observation_df[['customerid', 'stockcode']].drop_duplicates()

                # Join the stock level metrics with the product per customer df:
                customer_and_prod_lvl_metrics = stock_code_df_aggregated.merge(right=product_per_customer,
                                                               how='right',
                                                              on='stockcode')
                # Aggregate by customer and generate the customer order metrics/risk at product level:
                customer_risk_at_product_lvl = customer_and_prod_lvl_metrics.groupby(['customerid']).agg(
                                                avg_stockcode_issue_rate=('total_order_issue_rate_w_prod', 'mean'),
                                                max_stockcode_issue_rate=('total_order_issue_rate_w_prod', 'max'),
                                                total_orders_made_for_stock = ('total_n_orders_of_prod', 'sum')
                                            ).reset_index()
                logging.info("Success for creating the customer level risk for the products they buy.")
                return customer_risk_at_product_lvl                
            except Exception as e:
                raise CustomerChurnException(e, sys)
                

        def create_customer_lvl_agg_metrics(self, observation_df:pd.DataFrame, 
                                            customer_risk_at_product_lvl:pd.DataFrame,
                                           use_clipped:bool) -> pd.DataFrame:
            """
            Calculates the customer level metrics, takes in the customer risks associated with the products the customer have bought, merges both dataframe to create the final customer level features which holds the key metrics both at customer level and also at the product level that were bought by the customers.

            Parameters:
            ------------
            use_clipped: True-> uses the clipped column names else uses the raw column/unclipped column names
            """
            try:
                logging.info("Creating the customer level aggregated metrics.")
                if use_clipped:
                    qty_col = 'quantity_abs_clipped'
                    price_col = 'price_clipped'
                else:
                    qty_col = 'quantity_abs'
                    price_col = 'price'
                    
                    
                customer_data = observation_df[['invoice', 'quantity', 'invoicedate', 'customerid', 'country', 'order_issue',
                               qty_col, price_col]].copy()
                # Create customer level features:
                customer_data['total_amt_spent'] = customer_data['quantity'] * customer_data[price_col]
                customer_level_metrics = customer_data.groupby(by='customerid').agg(
                                        # 1. Frequency: Total num of unique orders made per customer:
                                        order_frequency = ('invoice', 'nunique'),
                                        # 2. Monetary: Total amount spent by each customer:
                                        total_monetary_value = ('total_amt_spent', 'sum'),
                                        # 3. Total quantity(abs value): involves purchased and had intentions to acquire:
                                        total_quantity_abs = (qty_col, 'sum'),
                                        # 4. Total issues in the orders made (order cancellation or products returned):
                                        total_order_issues = ('order_issue', 'sum'),
                                        # 5. First and last purchase made:
                                        first_purchase = ('invoicedate', 'min'),
                                        last_purchase = ('invoicedate', 'max'),
                                        # 6. Country: most probably has one country per customer but still a condition is defined:
                                        country = ('country', lambda c: c.mode()[0] if not c.mode().empty else 'Unspecified')
                                    ).reset_index()
                customer_level_metrics['avg_order_value'] = (customer_level_metrics['total_monetary_value'] 
                                                             / customer_level_metrics['order_frequency'])
                customer_level_metrics['avg_quantity_per_order'] = (customer_level_metrics['total_quantity_abs'] 
                                                                    / customer_level_metrics['order_frequency'])
                customer_level_metrics['customer_order_issue_rate'] = (customer_level_metrics['total_order_issues'] 
                                                                       / customer_level_metrics['order_frequency'])
                customer_level_metrics['recency'] = (self.churn_cutoff_date - customer_level_metrics['last_purchase']).dt.days
                customer_level_metrics['tenure'] = (self.churn_cutoff_date - customer_level_metrics['first_purchase']).dt.days

                # Dropping the columns that are not needed anymore:
                customer_level_metrics.drop(columns=['first_purchase', 'last_purchase'], inplace=True)

                # Merging two dfs to create the final customer level features: df for stockcode level metrics and customer level metrics:
                final_customer_lvl_features = customer_level_metrics.merge(customer_risk_at_product_lvl,
                                                 on='customerid',
                                                 how='left')
                logging.info("Success in creating the customer level aggregated metrics.")
                return final_customer_lvl_features               
            except Exception as e:
                raise CustomerChurnException(e, sys)


            def define_churn_label(self, df:pd.DataFrame, final_customer_lvl_features:pd.DataFrame) -> pd.DataFrame:
                """
                Takes in the final_customer_lvl_features df and labels churn:
                """
                try:
                    logging.info("Defining the churn label:")
                    logging.info(f"The observation window will be until: {self.observation_date}.")
                    logging.info(f"The churn window will be from: {self.observation_date + pd.DateOffset(days=1)} until:   {self.churn_cutoff_date}")
                    churn_df = df[(df['invoicedate'] > self.observation_date) & (df['invoicedate'] <= self.churn_cutoff_date)].copy()
                    logging.info(f"The min invoice date in churn df: {churn_df['invoicedate'].min()} and the max is: {churn_df['invoicedate'].max()}")
                    retained_customer_ids = set(churn_df['customerid'].unique())
                    logging.info(f"There are a total of: {len(retained_customer_ids)} customer IDs who made the transaction in churn window.")
                    final_customer_lvl_features['churn'] = (~final_customer_lvl_features['customerid'].isin(retained_customer_ids)).astype(int)
                    logging.info(f"The count for the churn label is:\n{final_customer_lvl_features['churn'].value_counts()}")
                    logging.info(f"The shape of the finalized df is:{final_customer_lvl_features.shape}")
                    logging.info("Success at defining the churn label.")
                    return final_customer_lvl_features                   
                except Exception as e:
                    raise CustomerChurnException(e, sys)

        def initiate_feature_engineering(self, branch_name:str):
            """
            Initiates the feature engineering pipeline:
            Parameters:
            -----------
            branch_name: clipped or unclipped
            clipped: data clipped at 99th quantile to clip the outlier values for the linear models
            unclipped: original data for tree models

            file_name: either training or testing data for the purpose of logging.
            """
            starting_time = time.perf_counter()
            logging.info("Feature engineering pipeline started:")
            use_clipped = False
            try:
                
                train_attr = f"{branch_name}_train_file_path"
                test_attr = f"{branch_name}_test_file_path"

                train_path = getattr(self, train_attr)
                test_path = getattr(self, test_attr)

                # Load the data:
                train_df = pd.read_parquet(train_path)
                test_df = pd.read_parquet(test_path)
                logging.info(f"Reading {branch_name} training and testing data completed.")
                logging.info(f"Training {branch_name} data has a shape of: {train_df.shape}")
                logging.info(f"Testing {branch_name} data has a shape of: {test_df.shape}")
                
                use_clipped = (branch_name == 'clipped')

                # 1. Create the observation records:
                train_df_obs = self.create_observation_df(df=train_df)
                test_df_obs = self.create_observation_df(df=test_df)

                # 2. Create the stock-code level metrics only using training data to avoid data leakage:
                train_stock_code_metrics_df = self.create_stockcode_lvl_metrics(observation_df=train_df_obs)
                

                # 3. Create customer risk at product/stock-code level:
                train_customer_risk_at_product_lvl = self.create_customer_risk_at_product_lvl(observation_df=train_df_obs,                                                                                    stock_code_df_aggregated=train_stock_code_metrics_df)
                test_customer_risk_at_product_lvl = self.create_customer_risk_at_product_lvl(observation_df=test_df_obs,                                                                                  stock_code_df_aggregated=train_stock_code_metrics_df)

                # 4. Create customer level aggregated metrics:
                train_final_customer_lvl_features = self.create_customer_lvl_agg_metrics(observation_df=train_df_obs,                                                                                customer_risk_at_product_lvl=train_customer_risk_at_product_lvl,
                                                                                    use_clipped=use_clipped)
                test_final_customer_lvl_features = self.create_customer_lvl_agg_metrics(observation_df=test_df_obs,                                                                                    customer_risk_at_product_lvl=test_customer_risk_at_product_lvl,
                                                                                    use_clipped=use_clipped)

                # 5. Define the churn:
                train_final_customer_lvl_features = self.define_churn_label(df=train_df,                                                                           final_customer_lvl_features=train_final_customer_lvl_features)
                test_final_customer_lvl_features = self.define_churn_label(df=test_df,                                                                          final_customer_lvl_features=test_final_customer_lvl_features)

            
                # Making the directory for the clipped data:
                os.makedirs(os.path.dirname(self.feature_engineering_config.clipped_train_file_path), exist_ok=True)
                os.makedirs(os.path.dirname(self.feature_engineering_config.unclipped_train_file_path), exist_ok=True)
                if use_clipped:
                    clipped_train_file_path = self.feature_engineering_config.clipped_train_file_path
                    clipped_test_file_path = self.feature_engineering_config.clipped_test_file_path
                    train_final_customer_lvl_features.to_parquet(clipped_train_file_path)
                    test_final_customer_lvl_features.to_parquet(clipped_test_file_path)
                    logging.info(f"Clipped training data saved to: {clipped_train_file_path}")
                    logging.info(f"Clipped testing data saved to: {clipped_test_file_path}")
                else:
                    unclipped_train_file_path = self.feature_engineering_config.unclipped_train_file_path
                    unclipped_test_file_path = self.feature_engineering_config.unclipped_test_file_path
                    train_final_customer_lvl_features.to_parquet(unclipped_train_file_path)
                    test_final_customer_lvl_features.to_parquet(unclipped_test_file_path)
                    logging.info(f"Unclipped training data saved to: {unclipped_train_file_path}")
                    logging.info(f"Unclipped testing data saved to: {unclipped_test_file_path}")

                # Generate the artifacts:
                if use_clipped:
                    feature_engineering_artifacts = FeatureEngineeringArtifacts(
                    clipped_training_file_path = clipped_train_file_path,
                    clipped_testing_file_path = clipped_test_file_path)
                else:
                    feature_engineering_artifacts = FeatureEngineeringArtifacts(
                        unclipped_training_file_path = unclipped_train_file_path,
                        unclipped_testing_file_path = unclipped_test_file_path
                    )
                ending_time = time.perf_counter()
                execution_time = round((ending_time - starting_time)/60, 3)
                logging.info(f"Feature Engineering Artifacts:\n{feature_engineering_artifacts}\n")
                logging.info(f"Feature Engineering Completed | Total Execution Time: {execution_time} min.")                
            except Exception as e:
                raise CustomerChurnException(e, sys)